In [49]:
import json
import joblib
import numpy as np
import os
import pandas as pd     
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, cohen_kappa_score


#import all algorithms
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.linear_model import SGDClassifier
#xgboost is also classfication model but using xgboost library,
#  and it is combination of decision tree and gradient boosting
from xgboost import XGBClassifier

import pickle

In [ ]:
#load the intent.json file
#we are using encoding utf-8 because it allow us to read special character with diffrent language.

with open('intents.json', encoding='utf-8') as f:
     data = json.load(f)

#store the patterns and tags in separate lists
#the patterns are the sentences that the user will input 
# and the tags are the labels that we want to predict

#sentence=== x:- it is user input and it will be checked with the patterns in the intent.json file 
# and then it will return the tag as output

#label=== y:- because it is the output that we want to predict and it is the tag in the intent.json file
#we check tag predicted by the model with the tag in the intent.json file and give response accordingly

sentences = []
labels = []
for intent in data['intents']:
   
    for pattern in intent['patterns']:
        sentences.append(pattern)
        labels.append(intent['tag'])


print("total sentences: ", len(sentences))
print("total labels: ", len(labels))

total sentences:  411
total labels:  411


In [66]:
#train test split
#stratify is used to ensure that the distribution of labels in the train and test sets is the same as in the original dataset
#not using stratify will distribute the labels randomly and may lead to an imbalanced distribution of..... 
# labels in the train and test sets, which can affect the performance of the model
x_train, x_test, y_train, y_test = train_test_split(sentences, labels, test_size=0.2, random_state=42, stratify=labels)


print(f"Train text samples: {len(x_train)}")
print(f"Test text samples: {len(x_test)}")
print()


Train text samples: 328
Test text samples: 83



In [67]:
#tfidf vectorizer is used to convert the text(sentence) data into numerical data 
#One-Hot Encoding is used to encode categorical data (not text sentences) into binary (0/1) format.
vectorizer = TfidfVectorizer()
x_train_tfidf = vectorizer.fit_transform(x_train)
x_test_tfidf = vectorizer.transform(x_test)
print("trainin data shape: ", x_train_tfidf.shape)
print("testing data shape: ", x_test_tfidf.shape)


trainin data shape:  (328, 375)
testing data shape:  (83, 375)


In [68]:
#eveluate the model using different algorithms and compare their performance
def evelute_func(y_test,y_pred):
    acc=accuracy_score(y_test,y_pred)
    print("accurcy",acc)
    cm=confusion_matrix(y_test,y_pred)
    print("confusion metrx",cm)
    kp=cohen_kappa_score(y_test,y_pred)
    print("Kappa scor",kp)
    cr=classification_report(y_test,y_pred)
    print("classification report",cr)
    err=1-acc
    print("error rate",err)

In [69]:
# 1. Multinomial Naive Bayes
nb = MultinomialNB()
nb.fit(x_train_tfidf, y_train)
y_pred = nb.predict(x_test_tfidf)
evelute_func(y_test, y_pred)

accurcy 0.6867469879518072
confusion metrx [[9 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [3 4 0 0 0 0 0 0 0 0 0 0 1 0]
 [0 0 5 0 0 0 0 0 0 0 0 0 0 0]
 [1 0 0 6 0 0 0 0 0 0 0 0 0 0]
 [2 2 0 1 1 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 6 0 0 0 0 0 0 0 0]
 [2 0 0 0 0 0 3 0 0 0 0 0 0 0]
 [1 1 0 1 0 0 0 0 0 0 0 0 0 1]
 [3 0 0 0 0 0 0 0 4 0 0 0 0 0]
 [1 1 0 0 0 0 0 0 0 1 0 0 0 0]
 [1 1 0 0 0 0 0 0 0 0 1 0 1 0]
 [0 0 0 0 0 0 0 0 0 0 0 7 0 0]
 [2 0 0 0 0 0 0 0 0 0 0 0 4 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 6]]
Kappa scor 0.6570247933884297
classification report                    precision    recall  f1-score   support

     crop_disease       0.36      1.00      0.53         9
       fertilizer       0.44      0.50      0.47         8
          goodbye       1.00      1.00      1.00         5
government_scheme       0.75      0.86      0.80         7
         greeting       1.00      0.17      0.29         6
       irrigation       1.00      1.00      1.00         6
     market_price       1.00      0.60      0.75        

In [70]:
#logestic regression
lr = LogisticRegression()
lr.fit(x_train_tfidf, y_train)
y_pred_lr = lr.predict(x_test_tfidf)
evelute_func(y_test,y_pred_lr)  

accurcy 0.7831325301204819
confusion metrx [[9 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [3 4 0 0 0 0 0 0 0 0 0 0 1 0]
 [0 0 5 0 0 0 0 0 0 0 0 0 0 0]
 [1 0 0 6 0 0 0 0 0 0 0 0 0 0]
 [2 2 0 1 1 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 6 0 0 0 0 0 0 0 0]
 [1 0 0 0 0 0 4 0 0 0 0 0 0 0]
 [1 0 0 0 0 0 0 2 0 0 0 0 0 1]
 [2 0 0 0 0 0 0 0 5 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 3 0 0 0 0]
 [1 0 0 0 0 0 0 0 0 0 2 0 1 0]
 [0 0 0 0 0 0 0 0 0 0 0 7 0 0]
 [1 0 0 0 0 0 0 0 0 0 0 0 5 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 6]]
Kappa scor 0.7637197532816701
classification report                    precision    recall  f1-score   support

     crop_disease       0.43      1.00      0.60         9
       fertilizer       0.67      0.50      0.57         8
          goodbye       1.00      1.00      1.00         5
government_scheme       0.86      0.86      0.86         7
         greeting       1.00      0.17      0.29         6
       irrigation       1.00      1.00      1.00         6
     market_price       1.00      0.80      0.89        

In [71]:
#svm
svm = SVC(kernel='linear')
svm.fit(x_train_tfidf, y_train)
y_pred_svm = svm.predict(x_test_tfidf)
evelute_func(y_test,y_pred_svm)

accurcy 0.8072289156626506
confusion metrx [[8 0 0 0 0 0 0 0 1 0 0 0 0 0]
 [1 1 0 0 0 0 0 2 2 0 0 1 1 0]
 [0 0 5 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 6 0 0 0 0 1 0 0 0 0 0]
 [0 0 0 0 4 0 0 0 2 0 0 0 0 0]
 [0 0 0 0 0 6 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 4 0 0 0 0 0 1 0]
 [0 0 0 0 0 0 0 2 1 0 0 0 0 1]
 [1 0 0 0 0 0 0 0 6 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 3 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 3 0 1 0]
 [0 0 0 0 0 0 0 0 0 0 0 7 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 6 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 6]]
Kappa scor 0.7913262099308611
classification report                    precision    recall  f1-score   support

     crop_disease       0.80      0.89      0.84         9
       fertilizer       1.00      0.12      0.22         8
          goodbye       1.00      1.00      1.00         5
government_scheme       1.00      0.86      0.92         7
         greeting       1.00      0.67      0.80         6
       irrigation       1.00      1.00      1.00         6
     market_price       1.00      0.80      0.89        

In [72]:
#random forest
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(x_train_tfidf, y_train)
y_pred_rf = rf.predict(x_test_tfidf)
evelute_func(y_test,y_pred_rf)

accurcy 0.7590361445783133
confusion metrx [[6 0 0 0 1 0 0 0 2 0 0 0 0 0]
 [1 1 0 0 2 0 0 2 0 0 0 1 1 0]
 [0 0 5 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 5 1 0 0 0 0 0 1 0 0 0]
 [0 0 0 1 5 0 0 0 0 0 0 0 0 0]
 [0 1 0 0 0 5 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 5 0 0 0 0 0 0 0]
 [0 0 0 0 1 0 0 3 0 0 0 0 0 0]
 [3 0 0 0 0 0 0 0 4 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 3 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 3 0 1 0]
 [0 0 0 0 0 0 0 0 0 0 0 7 0 0]
 [0 0 0 0 0 0 1 0 0 0 0 0 5 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 6]]
Kappa scor 0.7396078431372549
classification report                    precision    recall  f1-score   support

     crop_disease       0.60      0.67      0.63         9
       fertilizer       0.50      0.12      0.20         8
          goodbye       1.00      1.00      1.00         5
government_scheme       0.83      0.71      0.77         7
         greeting       0.50      0.83      0.62         6
       irrigation       1.00      0.83      0.91         6
     market_price       0.83      1.00      0.91        

In [73]:
#use k cross validation to find the best k for KNN for get the best performance of KNN
from sklearn.model_selection import cross_val_score

for k in range(1, 21):
    knn = KNeighborsClassifier(n_neighbors=k)
    scores = cross_val_score(knn, x_train_tfidf, y_train, cv=5)
    print(k, scores.mean())

    #take the highest score and use that k to train the KNN model and evaluate it
    #here k=6 is the best k for KNN 0.4608....
#even k=8 is best we use k=7 because k=9 is even and it may lead to tie in the voting of KNN and we want to avoid that tie in voting
knn = KNeighborsClassifier(n_neighbors=7)
knn.fit(x_train_tfidf, y_train)
y_pred_knn = knn.predict(x_test_tfidf)
evelute_func(y_test,y_pred_knn)

1 0.576037296037296
2 0.5582750582750583
3 0.6064801864801865
4 0.6338461538461538
5 0.6003263403263404
6 0.6187878787878789
7 0.6158508158508159
8 0.5883916083916085
9 0.5760372960372961
10 0.5793006993006993
11 0.5731002331002332
12 0.567039627039627
13 0.564009324009324
14 0.5487645687645688
15 0.5395804195804196
16 0.5365501165501165
17 0.5274592074592074
18 0.5243822843822844
19 0.5243822843822844
20 0.5274125874125875
accurcy 0.6867469879518072
confusion metrx [[8 0 0 0 0 1 0 0 0 0 0 0 0 0]
 [0 2 0 0 0 4 0 1 0 0 0 0 1 0]
 [0 0 5 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 6 0 1 0 0 0 0 0 0 0 0]
 [0 2 0 0 1 3 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 6 0 0 0 0 0 0 0 0]
 [1 0 0 0 0 0 3 0 0 0 1 0 0 0]
 [0 0 0 1 0 1 0 2 0 0 0 0 0 0]
 [2 1 0 0 0 1 0 0 3 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 3 0 0 0 0]
 [0 0 0 0 0 1 0 0 0 0 2 0 1 0]
 [0 0 0 0 0 0 0 0 1 0 0 5 1 0]
 [0 1 0 0 0 0 0 0 0 0 0 0 5 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 6]]
Kappa scor 0.6606384651674791
classification report                    precision    recall  f1

In [74]:
#decision tree
dt = DecisionTreeClassifier(random_state=42)
dt.fit(x_train_tfidf, y_train)
y_pred_dt = dt.predict(x_test_tfidf)
evelute_func(y_test,y_pred_dt)

accurcy 0.7108433734939759
confusion metrx [[5 0 0 0 3 0 0 0 0 0 0 0 1 0]
 [0 0 0 0 3 0 0 2 0 1 0 1 1 0]
 [0 0 5 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 4 2 0 0 0 0 0 1 0 0 0]
 [0 0 0 1 5 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 2 4 0 0 0 0 0 0 0 0]
 [0 0 0 0 1 0 4 0 0 0 0 0 0 0]
 [0 0 0 0 1 0 0 3 0 0 0 0 0 0]
 [2 0 0 0 0 0 0 0 4 0 0 0 1 0]
 [0 0 0 0 0 0 0 0 0 3 0 0 0 0]
 [0 0 0 0 1 0 0 0 0 0 3 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 7 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 6 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 6]]
Kappa scor 0.688360450563204
classification report                    precision    recall  f1-score   support

     crop_disease       0.71      0.56      0.62         9
       fertilizer       0.00      0.00      0.00         8
          goodbye       1.00      1.00      1.00         5
government_scheme       0.80      0.57      0.67         7
         greeting       0.28      0.83      0.42         6
       irrigation       1.00      0.67      0.80         6
     market_price       1.00      0.80      0.89         

In [75]:
#gradient boosting
gb = GradientBoostingClassifier(random_state=42)
gb.fit(x_train_tfidf, y_train)
y_pred_gb = gb.predict(x_test_tfidf)
evelute_func(y_test,y_pred_gb)

accurcy 0.7349397590361446
confusion metrx [[5 0 0 0 0 0 0 0 4 0 0 0 0 0]
 [0 2 0 0 0 0 0 2 2 0 0 1 1 0]
 [0 0 5 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 4 0 0 0 0 2 0 1 0 0 0]
 [0 0 0 0 4 0 0 0 2 0 0 0 0 0]
 [0 0 0 0 0 6 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 5 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 2 2 0 0 0 0 0]
 [2 0 0 0 0 0 0 0 4 0 0 0 1 0]
 [0 0 0 0 0 0 0 0 0 3 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 4 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 7 0 0]
 [1 0 0 0 0 0 0 0 1 0 0 0 4 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 6]]
Kappa scor 0.7133887929681368
classification report                    precision    recall  f1-score   support

     crop_disease       0.62      0.56      0.59         9
       fertilizer       1.00      0.25      0.40         8
          goodbye       1.00      1.00      1.00         5
government_scheme       1.00      0.57      0.73         7
         greeting       1.00      0.67      0.80         6
       irrigation       1.00      1.00      1.00         6
     market_price       1.00      1.00      1.00        

In [76]:
#sgd
sgd = SGDClassifier(random_state=42)
sgd.fit(x_train_tfidf, y_train)
y_pred_sgd = sgd.predict(x_test_tfidf)    
evelute_func(y_test,y_pred_sgd)


accurcy 0.8313253012048193
confusion metrx [[8 0 0 0 1 0 0 0 0 0 0 0 0 0]
 [0 2 0 0 2 0 0 2 0 0 0 1 1 0]
 [0 0 5 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 6 1 0 0 0 0 0 0 0 0 0]
 [0 2 0 1 3 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 6 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 5 0 0 0 0 0 0 0]
 [0 0 0 0 1 0 0 3 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 7 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 3 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 3 0 1 0]
 [0 0 0 0 0 0 0 0 0 0 0 7 0 0]
 [0 1 0 0 0 0 0 0 0 0 0 0 5 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 6]]
Kappa scor 0.8175824175824176
classification report                    precision    recall  f1-score   support

     crop_disease       1.00      0.89      0.94         9
       fertilizer       0.40      0.25      0.31         8
          goodbye       1.00      1.00      1.00         5
government_scheme       0.86      0.86      0.86         7
         greeting       0.38      0.50      0.43         6
       irrigation       1.00      1.00      1.00         6
     market_price       1.00      1.00      1.00        

In [77]:
# XGBoost
from sklearn.preprocessing import LabelEncoder

# Encode labels once
le = LabelEncoder()
y_train_encoded = le.fit_transform(y_train)
y_test_encoded = le.transform(y_test)

# Train and predict
xgb = XGBClassifier(random_state=42)
xgb.fit(x_train_tfidf, y_train_encoded)
# Inverse transform predictions to original labels like 'greeting', 'goodbye', etc.
y_pred_xgb = le.inverse_transform(xgb.predict(x_test_tfidf))
evelute_func(y_test, y_pred_xgb)

accurcy 0.5662650602409639
confusion metrx [[5 1 0 2 0 0 0 0 0 0 0 0 1 0]
 [0 0 0 2 0 0 0 2 0 1 0 1 2 0]
 [0 0 3 2 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 5 0 0 1 0 0 0 1 0 0 0]
 [0 0 0 6 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 2 0 4 0 0 0 0 0 0 0 0]
 [0 0 0 2 0 0 3 0 0 0 0 0 0 0]
 [0 0 0 2 0 0 0 2 0 0 0 0 0 0]
 [2 0 0 2 0 0 0 0 3 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 3 0 0 0 0]
 [0 0 0 2 0 0 0 0 0 0 2 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 7 0 0]
 [0 0 0 0 1 0 1 0 0 0 0 0 4 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 6]]
Kappa scor 0.5304839723444374
classification report                    precision    recall  f1-score   support

     crop_disease       0.71      0.56      0.62         9
       fertilizer       0.00      0.00      0.00         8
          goodbye       1.00      0.60      0.75         5
government_scheme       0.19      0.71      0.29         7
         greeting       0.00      0.00      0.00         6
       irrigation       1.00      0.67      0.80         6
     market_price       0.60      0.60      0.60        

In [78]:
#find the best model and save the best model and vectorizer using joblib
results = {
    "Naive Bayes"         : accuracy_score(y_test, y_pred),
    "Logistic Regression" : accuracy_score(y_test, y_pred_lr),
    "SVM"                 : accuracy_score(y_test, y_pred_svm),
    "Random Forest"       : accuracy_score(y_test, y_pred_rf),
    "KNN"                 : accuracy_score(y_test, y_pred_knn),
    "Decision Tree"       : accuracy_score(y_test, y_pred_dt),
    "Gradient Boosting"   : accuracy_score(y_test, y_pred_gb),
    "SGD"                 : accuracy_score(y_test, y_pred_sgd),
}

best_name = max(results, key=results.get)
print(f"Best model: {best_name} — {results[best_name]*100:.1f}%")



Best model: SGD — 83.1%


In [79]:
#save the best model
best_models = {
    "Naive Bayes": nb, "Logistic Regression": lr,
    "SVM": svm, "Random Forest": rf, "KNN": knn,
    "Decision Tree": dt, "Gradient Boosting": gb, "SGD": sgd
}
best_clf = best_models[best_name]
joblib.dump(best_clf, "model.pkl")


#save the vectorizer because we need to use the same vectorizer to 
# transform the user input before making predictions with the model
joblib.dump(vectorizer,"vectorizer.pkl")
print(f"Saved best model: {best_name} with vectorizer file")


Saved best model: SGD with vectorizer file
